# decisionrl — Quickstart

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DenisDrobyshev/decisionrl/blob/main/examples/quickstart.ipynb)

A 5-minute tour of the [`decisionrl`](https://github.com/DenisDrobyshev/decisionrl) library:
train PPO on CartPole, plot the learning curve, evaluate, save/load, and solve a GridWorld with tabular Q-Learning.


## Install


In [ ]:
%pip install -q git+https://github.com/DenisDrobyshev/decisionrl.git matplotlib


## Train PPO on CartPole
Every agent captures metrics through a `Logger`; `HistoryLogger` keeps them in memory for plotting.


In [ ]:
from decisionrl.algorithms import PPO
from decisionrl.envs import CartPole
from decisionrl.utils import HistoryLogger, set_seed

set_seed(0)
log = HistoryLogger()
agent = PPO(CartPole(), n_steps=1024, batch_size=64, n_epochs=10, seed=0, logger=log)
agent.learn(total_steps=40_000)


### Plot the learning curve


In [ ]:
import matplotlib.pyplot as plt
steps, returns = log.curve('rollout/ep_return_mean')
plt.figure(figsize=(7, 4))
plt.plot(steps, returns)
plt.xlabel('environment steps'); plt.ylabel('episode return')
plt.title('PPO on CartPole (max 500)'); plt.grid(True); plt.show()


## Evaluate, save and load


In [ ]:
from decisionrl.training import evaluate_policy
mean, std = evaluate_policy(agent, CartPole(), n_episodes=20)
print(f'return = {mean:.1f} +/- {std:.1f}')

agent.save('ppo_cartpole.pt')
agent = PPO.load('ppo_cartpole.pt', env=CartPole())


## Tabular Q-Learning on GridWorld
A dependency-free tabular method that recovers the optimal navigation policy.


In [ ]:
from decisionrl.algorithms import QLearning
from decisionrl.envs import GridWorld

set_seed(0)
env = GridWorld(rows=4, cols=4, start=(0, 0), goal=(3, 3))
q = QLearning(env, learning_rate=0.2, seed=0)
q.learn(total_steps=30_000)

arrows = {0: '^', 1: '>', 2: 'v', 3: '<'}
for r in range(env.rows):
    row = []
    for c in range(env.cols):
        row.append('G' if (r, c) == env.goal else arrows[q.predict(r * env.cols + c)])
    print(' '.join(row))


## Where next?

- **Algorithms**: DQN/C51/QR-DQN, A2C, SAC/TD3/DDPG, SAC-Discrete, offline TD3+BC/IQL
- **CLI**: `decisionrl train ppo CartPole --steps 50000 --progress`
- **Docs**: https://denisdrobyshev.github.io/decisionrl/
